In [1]:
# Imports

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tabulate import tabulate

# Load data 
# MAE-trained models
icrnn = np.load("M1_iCRNN_training_results_MAE.npz")
crnn  = np.load("M1_CRNN_training_results_MAE.npz")
# MSE-trained models
icrnn_mse = np.load("M1_iCRNN_training_results_MSE.npz")
crnn_mse  = np.load("M1_CRNN_training_results_MSE.npz")

# Labels of noise levels (no noise, low noise, high noise)
noise_labels = ["Denoised", "Low noise", "High noise"]

# Extract # of noise levels, # of sets of kinetic constants, # of training epochs
n_noise, n_ks, n_epoch = icrnn["train_loss_all"].shape

# Masks for finished training runs
icrnn_done = icrnn["done_mask"] > 0.5
crnn_done  = crnn["done_mask"] > 0.5
crnn_failed = crnn["failed_mask"] > 0.5 if "failed_mask" in crnn else np.zeros_like(crnn_done, dtype=bool)

# Plotting functions
IND_ALPHA  = 0.05           # Alpha for individual loss lines
MEAN_ALPHA = 1.0            # Alpha for mean loss lines

IND_LW     = 0.3            # Linewidth for individual loss lines
MEAN_LW    = 1.5            # Linewidth for mean loss lines

icrnn_colour = "tab:blue"   # Colour for icrnn loss lines
crnn_colour = "tab:red"     # Colour for crnn loss lines

def load_method(data, label):
    """Prepare arrays, mask unfinished runs with NaN."""
    train_loss_all = data["train_loss_all"].copy()
    val_loss_all   = data["val_loss_all"].copy()
    time_all       = data["time_all"].copy()
    p_all          = data["p_all"].copy()
    done_mask      = data["done_mask"] > 0.5

    n_done = int(done_mask.sum())
    n_total = done_mask.size
    print(f"{label}: {n_done}/{n_total} (noise, k) pairs trained so far.")

    if "failed_mask" in data:
        failed = data["failed_mask"] > 0.5
        n_failed = int(failed.sum())
        if n_failed > 0:
            print(f"  ( {n_failed} failed run(s) – excluded from means)")

    # mask unfinished runs
    not_done = ~done_mask
    train_loss_all[not_done, :] = np.nan
    val_loss_all[not_done, :]   = np.nan
    time_all[not_done, :]       = np.nan
    p_all[not_done, :]          = np.nan
    return train_loss_all, val_loss_all, time_all, p_all, done_mask

# Load all four results
train_i,     val_i,     time_i,     p_i,     done_i     = load_method(icrnn,     "iCRNN (MAE)")
train_c,     val_c,     time_c,     p_c,     done_c     = load_method(crnn,      "CRNN  (MAE)")
train_i_mse, val_i_mse, time_i_mse, p_i_mse, done_i_mse = load_method(icrnn_mse, "iCRNN (MSE)")
train_c_mse, val_c_mse, time_c_mse, p_c_mse, done_c_mse = load_method(crnn_mse,  "CRNN  (MSE)")

both_done = int((done_i & done_c).sum())
print(f"{both_done} (noise, k) pairs completed in ALL methods.\n")


# Geometric mean helper
def geomean(a, axis=None):
    """Geometric mean ignoring NaNs.

    Entries must be strictly positive; zeros and negatives raise an error,
    since log is undefined there. NaN entries are ignored.
    """
    a = np.asarray(a, dtype=float)
    mask = np.isfinite(a)
    if not np.all(a[mask] > 0):
        raise ValueError("geomean: non-positive entries are not allowed")
    with np.errstate(invalid="ignore", divide="ignore"):
        log_a = np.where(mask, np.log(a), np.nan)
    return np.exp(np.nanmean(log_a, axis=axis))


# Plotting
def plot_loss_vs_epoch(train_i, val_i, train_c, val_c,
                        loss_name, out_path):
    """1x3 figure: columns = noise levels, single panel per column
    showing train (solid) and val (dashed) losses vs epoch.
    Means are geometric means over k."""
    epochs = np.arange(1, n_epoch + 1)

    mean_train_i = geomean(train_i, axis=1)
    mean_val_i   = geomean(val_i,   axis=1)
    mean_train_c = geomean(train_c, axis=1)
    mean_val_c   = geomean(val_c,   axis=1)

    fig, axes = plt.subplots(1, n_noise, figsize=(5 * n_noise, 4), squeeze=False)

    for i in range(n_noise):
        ax = axes[0, i]

        # Individual runs
        for k in range(n_ks):
            ax.plot(epochs, train_c[i, k], color=crnn_colour,  alpha=IND_ALPHA, lw=IND_LW)
            ax.plot(epochs, val_c[i, k],   color=crnn_colour,  alpha=IND_ALPHA, lw=IND_LW, ls="--")
            ax.plot(epochs, train_i[i, k], color=icrnn_colour, alpha=IND_ALPHA, lw=IND_LW)
            ax.plot(epochs, val_i[i, k],   color=icrnn_colour, alpha=IND_ALPHA, lw=IND_LW, ls="--")

        # Means (geometric)
        ax.plot(epochs, mean_train_c[i], color=crnn_colour,  alpha=MEAN_ALPHA, lw=MEAN_LW, ls="-",  label="CRNN — train")
        ax.plot(epochs, mean_val_c[i],   color=crnn_colour,  alpha=MEAN_ALPHA, lw=MEAN_LW, ls="--", label="CRNN — val")
        ax.plot(epochs, mean_train_i[i], color=icrnn_colour, alpha=MEAN_ALPHA, lw=MEAN_LW, ls="-",  label="iCRNN — train")
        ax.plot(epochs, mean_val_i[i],   color=icrnn_colour, alpha=MEAN_ALPHA, lw=MEAN_LW, ls="--", label="iCRNN — val")

        ax.set_yscale("log")
        ax.set_ylim(5e-4, 1.0)
        ax.set_title(f"{noise_labels[i]}")
        if i == 0:
            ax.set_ylabel(f"{loss_name} loss")
        ax.legend(fontsize=8)

    for i in range(n_noise):
        axes[0, i].set_xlabel("Epoch")

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")


def plot_loss_vs_time(train_i, val_i, time_i,
                       train_c, val_c, time_c,
                       loss_name, out_path):
    """1x3 figure: columns = noise levels, single panel per column,
    loss vs wall-clock time.

    Loss means are geometric (over k); time means are arithmetic, since
    wall-clock time is an additive cost.
    """
    mean_train_i = geomean(train_i, axis=1)
    mean_val_i   = geomean(val_i,   axis=1)
    mean_time_i  = np.nanmean(time_i, axis=1)

    mean_train_c = geomean(train_c, axis=1)
    mean_val_c   = geomean(val_c,   axis=1)
    mean_time_c  = np.nanmean(time_c, axis=1)

    fig, axes = plt.subplots(1, n_noise, figsize=(5 * n_noise, 4), squeeze=False)

    for i in range(n_noise):
        ax = axes[0, i]

        # Individual runs (validation loss only; train lines commented out)
        for k in range(n_ks):
            # ax.plot(time_c[i, k], train_c[i, k], color=crnn_colour,  alpha=IND_ALPHA, lw=IND_LW, ls="--")
            ax.plot(time_c[i, k], val_c[i, k],   color=crnn_colour,  alpha=IND_ALPHA, lw=IND_LW, ls="-")
            # ax.plot(time_i[i, k], train_i[i, k], color=icrnn_colour, alpha=IND_ALPHA, lw=IND_LW, ls="--")
            ax.plot(time_i[i, k], val_i[i, k],   color=icrnn_colour, alpha=IND_ALPHA, lw=IND_LW, ls="-")

        # Means (geometric for loss, arithmetic for time)
        # ax.plot(mean_time_c[i], mean_train_c[i], color=crnn_colour,  alpha=MEAN_ALPHA, lw=MEAN_LW, ls="--", label="CRNN — train")
        ax.plot(mean_time_c[i], mean_val_c[i],   color=crnn_colour,  alpha=MEAN_ALPHA, lw=MEAN_LW, ls="-",  label="CRNN — val")
        # ax.plot(mean_time_i[i], mean_train_i[i], color=icrnn_colour, alpha=MEAN_ALPHA, lw=MEAN_LW, ls="--", label="iCRNN — train")
        ax.plot(mean_time_i[i], mean_val_i[i],   color=icrnn_colour, alpha=MEAN_ALPHA, lw=MEAN_LW, ls="-",  label="iCRNN — val")

        ax.set_yscale("log")
        ax.set_ylim(1e-5, 1.0)
        ax.set_xlim(0, 400)
        ax.set_title(f"{noise_labels[i]}")
        if i == 0:
            ax.set_ylabel(f"{loss_name} loss")
        ax.legend(fontsize=8)

    for i in range(n_noise):
        axes[0, i].set_xlabel("Wall-clock time (s)")

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")


# MAE figures
# plot_loss_vs_epoch(train_i,     val_i,     train_c,     val_c,
#                     loss_name="MAE",
#                     out_path="M1_MAE_vs_epoch.pdf")
plot_loss_vs_time(train_i,     val_i,     time_i,
                   train_c,     val_c,     time_c,
                   loss_name="MAE",
                   out_path="M1_MAE_vs_time.pdf")

# MSE figures
# plot_loss_vs_epoch(train_i_mse, val_i_mse, train_c_mse, val_c_mse,
#                     loss_name="MSE",
#                     out_path="M1_MSE_vs_epoch.pdf")
plot_loss_vs_time(train_i_mse, val_i_mse, time_i_mse,
                   train_c_mse, val_c_mse, time_c_mse,
                   loss_name="MSE",
                   out_path="M1_MSE_vs_time.pdf")


# Compact summary: geometric-mean final loss and arithmetic-mean runtime
def summary_row(method_name, train, val, time, done, noise_idx):
    """Final-epoch loss (geometric mean over k) and runtime (arithmetic mean).

    Runs that did not finish, or whose final loss is non-finite, are
    excluded from the corresponding mean. The number of contributing runs
    is reported alongside.
    """
    tl = train[noise_idx, :, -1]   # final-epoch train loss per k
    vl = val[noise_idx,   :, -1]   # final-epoch val loss per k
    rt = time[noise_idx,  :, -1]   # total runtime per k

    # Restrict to finished runs with finite values
    tl_fin = tl[done[noise_idx] & np.isfinite(tl)]
    vl_fin = vl[done[noise_idx] & np.isfinite(vl)]
    rt_fin = rt[done[noise_idx] & np.isfinite(rt)]

    n_vl = vl_fin.size

    def sci(x):
        return f"{x:.1e}"

    return {
        "Method":     method_name,
        "Noise":      noise_labels[noise_idx],
        "TrainLoss":  sci(geomean(tl_fin)),          # geometric mean
        "ValLoss":    sci(geomean(vl_fin)),          # geometric mean
        "Runtime":    f"{np.mean(rt_fin):.1f} s",    # arithmetic mean
        "n":          f"{n_vl}",
    }


rows = []
methods = [
    ("iCRNN (MAE)", train_i,     val_i,     time_i,     done_i),
    ("CRNN  (MAE)", train_c,     val_c,     time_c,     done_c & ~crnn_failed),
    ("iCRNN (MSE)", train_i_mse, val_i_mse, time_i_mse, done_i_mse),
    ("CRNN  (MSE)", train_c_mse, val_c_mse, time_c_mse, done_c_mse & ~crnn_failed),
]

for noise_idx in range(n_noise):
    for name, tr, vl_, tm, dn in methods:
        rows.append(summary_row(name, tr, vl_, tm, dn, noise_idx))

df = pd.DataFrame(rows)
print("\nCompact summary (geometric mean for losses, arithmetic mean for runtime):\n")
print(tabulate(rows, headers="keys", tablefmt="pipe", showindex=False))

# Optional LaTeX export
# with open("summary.tex", "w") as f:
#     f.write(df.to_latex(index=False))

iCRNN (MAE): 300/300 (noise, k) pairs trained so far.
CRNN  (MAE): 300/300 (noise, k) pairs trained so far.
  ( 2 failed run(s) – excluded from means)
iCRNN (MSE): 300/300 (noise, k) pairs trained so far.
CRNN  (MSE): 300/300 (noise, k) pairs trained so far.
  ( 3 failed run(s) – excluded from means)
300 (noise, k) pairs completed in ALL methods.

Saved M1_MAE_vs_time.pdf


/var/folders/dd/g4rx_qqd5_x8cdgdtcrzswg00000gp/T/ipykernel_45659/3728461876.py:182: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.tight_layout()
/var/folders/dd/g4rx_qqd5_x8cdgdtcrzswg00000gp/T/ipykernel_45659/3728461876.py:183: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(out_path, dpi=300, bbox_inches="tight")


Saved M1_MSE_vs_time.pdf

Compact summary (geometric mean for losses, arithmetic mean for runtime):

| Method      | Noise      |   TrainLoss |   ValLoss | Runtime   |   n |
|:------------|:-----------|------------:|----------:|:----------|----:|
| iCRNN (MAE) | Denoised   |     0.0077  |   0.0083  | 90.9 s    | 100 |
| CRNN  (MAE) | Denoised   |     0.0063  |   0.0065  | 213.2 s   |  98 |
| iCRNN (MSE) | Denoised   |     0.00015 |   0.00017 | 92.2 s    | 100 |
| CRNN  (MSE) | Denoised   |     8.7e-05 |   0.0001  | 241.6 s   |  96 |
| iCRNN (MAE) | Low noise  |     0.017   |   0.019   | 90.5 s    | 100 |
| CRNN  (MAE) | Low noise  |     0.012   |   0.013   | 202.6 s   | 100 |
| iCRNN (MSE) | Low noise  |     0.00058 |   0.00067 | 93.2 s    | 100 |
| CRNN  (MSE) | Low noise  |     0.00036 |   0.00041 | 205.7 s   |  99 |
| iCRNN (MAE) | High noise |     0.026   |   0.028   | 90.8 s    | 100 |
| CRNN  (MAE) | High noise |     0.018   |   0.019   | 196.5 s   | 100 |
| iCRNN (MSE) | High no